# Загрузка и импорт библиотек

In [22]:
%%capture
!pip install datasets transformers evaluate seqeval

In [81]:
import ast
import torch
from datasets import Dataset, DatasetDict
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import pandas as pd

# Работа с датасетами

Первичная загрузка датасета

In [41]:
class LoadDataset:
    def __init__(self, train_path, val_path, test_path):
        self.train_path = train_path
        self.val_path = val_path
        self.test_path = test_path

        train_raw, val_raw, test_raw = self.__load_raws__()
        self.__make_2s__(train_raw)
        train_ds, val_ds, test_ds = self.__make_datasets__(train_raw, val_raw, test_raw)
        self.__make_dataset_dict__(train_ds, val_ds, test_ds)

    def __load_ner_file__(self, path):
        with open(path, "r", encoding="utf-8") as f:
            raw = f.read()
        data = ast.literal_eval(raw)
        return data

    def __load_raws__(self):
        train_raw = self.__load_ner_file__(self.train_path)
        val_raw = self.__load_ner_file__(self.val_path)
        test_raw = self.__load_ner_file__(self.test_path)

        return train_raw, val_raw, test_raw

    def __make_2s__(self, raw):
        self.id2label = {int(k): v for k, v in raw["vocab_ner_tags"].items()}
        self.label2id = {v: int(k) for k, v in raw["vocab_ner_tags"].items()}

    def __make_datasets__(self, train_raw, val_raw, test_raw):
        train_ds = Dataset.from_list(train_raw["sentencies"])
        val_ds = Dataset.from_list(val_raw["sentencies"])
        test_ds = Dataset.from_list(test_raw["sentencies"])

        return train_ds, val_ds, test_ds

    def __make_dataset_dict__(self, train_ds, val_ds, test_ds):
        self.dataset = DatasetDict({
            "train": train_ds,
            "validation": val_ds,
            "test": test_ds
        })

    def get_dataset(self):
        return self.dataset

    def get_id2label(self):
        return self.id2label

    def get_label2id(self):
        return self.label2id

Функция для токенизации датасета.

In [42]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

# Обучение и сравнение

## Формирование датасета и модели

Загрузка датасета

In [48]:
dataset_loader = LoadDataset("train.txt", "val.txt", "test.txt")

dataset = dataset_loader.get_dataset()
id2label = dataset_loader.get_id2label()
label2id = dataset_loader.get_label2id()

In [49]:
print(dataset)
print(dataset["train"][0])

print(id2label)
print(label2id)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 200
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 200
    })
})
{'tokens': ['Первые', 'дни', 'раскола', 'Эмурланна'], 'ner_tags': [0, 0, 0, 3]}
{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-LOC', 4: 'I-LOC', 5: 'B-RACE', 6: 'I-RACE'}
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-LOC': 3, 'I-LOC': 4, 'B-RACE': 5, 'I-RACE': 6}


Загрузка токенизатора и токенизация датасета для обучения

In [50]:
model_name = "DeepPavlov/rubert-base-cased"

In [51]:
%%capture
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [52]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

sample = tokenized_dataset["train"][0]
print(sample.keys())
print(sample["tokens"])
print(sample["ner_tags"])
print(tokenizer.convert_ids_to_tokens(sample["input_ids"]))
print(sample["labels"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

dict_keys(['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])
['Первые', 'дни', 'раскола', 'Эмурланна']
[0, 0, 0, 3]
['[CLS]', 'Первые', 'дни', 'раскола', 'Эм', '##ур', '##ланн', '##а', '[SEP]']
[-100, 0, 0, 0, 3, -100, -100, -100, -100]


In [53]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Функция для вычисления оценок при обучении

In [54]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                current_preds.append(id2label[int(pred_id)])
                current_labels.append(id2label[int(label_id)])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Загрузка модели

In [55]:
%%capture
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

untrained_model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                          

## Обучение

Формирование trainer

In [56]:
seqeval = evaluate.load("seqeval")

In [57]:
training_args = TrainingArguments(
    output_dir="./rubert-ner-malazan",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",
)

In [58]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [59]:
untrained_model_trainer = Trainer(
    model=untrained_model,
    args=training_args,
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Качество модели до обучения

In [61]:
untrained_model_metrics = untrained_model_trainer.evaluate(tokenized_dataset["test"])
print(untrained_model_metrics)

{'eval_loss': 1.9095262289047241, 'eval_model_preparation_time': 0.0031, 'eval_precision': 0.008984105044920525, 'eval_recall': 0.1625, 'eval_f1': 0.017026850032743943, 'eval_accuracy': 0.20702576112412177, 'eval_runtime': 0.8167, 'eval_samples_per_second': 244.881, 'eval_steps_per_second': 30.61}


Обучение

In [62]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.153223,0.035332,0.860000,0.914894,0.886598,0.992097
2,0.023794,0.025420,0.936842,0.946809,0.941799,0.994731
3,0.009370,0.027998,0.956989,0.946809,0.951872,0.995785
4,0.004095,0.027386,0.956522,0.936170,0.946237,0.994731
5,0.002590,0.027154,0.956989,0.946809,0.951872,0.995785


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=625, training_loss=0.038614492702484134, metrics={'train_runtime': 231.5605, 'train_samples_per_second': 21.593, 'train_steps_per_second': 2.699, 'total_flos': 86366564860752.0, 'train_loss': 0.038614492702484134, 'epoch': 5.0})

Результаты обучения

In [63]:
val_metrics = trainer.evaluate(tokenized_dataset["validation"])
print("Validation metrics:", val_metrics)

test_metrics = trainer.evaluate(tokenized_dataset["test"])
print("Test metrics:", test_metrics)

Validation metrics: {'eval_loss': 0.027983613312244415, 'eval_precision': 0.956989247311828, 'eval_recall': 0.9468085106382979, 'eval_f1': 0.9518716577540107, 'eval_accuracy': 0.9957850368809273, 'eval_runtime': 0.5548, 'eval_samples_per_second': 360.467, 'eval_steps_per_second': 45.058, 'epoch': 5.0}
Test metrics: {'eval_loss': 0.05247701704502106, 'eval_precision': 0.6818181818181818, 'eval_recall': 0.75, 'eval_f1': 0.7142857142857143, 'eval_accuracy': 0.9882903981264637, 'eval_runtime': 0.5475, 'eval_samples_per_second': 365.264, 'eval_steps_per_second': 45.658, 'epoch': 5.0}


Сравнение обученной и необученной моделей

In [64]:
trained_test_metrics = test_metrics

In [67]:
metrics_df = pd.DataFrame([
    {
        "model": "untrained_model",
        "precision": untrained_model_metrics["eval_precision"],
        "recall": untrained_model_metrics["eval_recall"],
        "f1": untrained_model_metrics["eval_f1"],
        "accuracy": untrained_model_metrics["eval_accuracy"],
    },
    {
        "model": "fine_tuned_model",
        "precision": trained_test_metrics["eval_precision"],
        "recall": trained_test_metrics["eval_recall"],
        "f1": trained_test_metrics["eval_f1"],
        "accuracy": trained_test_metrics["eval_accuracy"],
    }
])

print(metrics_df)

              model  precision  recall        f1  accuracy
0   untrained_model   0.008984  0.1625  0.017027  0.207026
1  fine_tuned_model   0.681818  0.7500  0.714286  0.988290


Сохранение обученной модели

In [66]:
trainer.save_model("./rubert-ner-malazan-best")
tokenizer.save_pretrained("./rubert-ner-malazan-best")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./rubert-ner-malazan-best/tokenizer_config.json',
 './rubert-ner-malazan-best/tokenizer.json')

## Пример применения моделей

Функция предсказания на уровне слов

In [83]:
def predict_word_labels(model, tokenizer, tokens, id2label, max_length=128):
    model.eval()
    device = model.device

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    word_ids = encoding.word_ids(batch_index=0)
    model_inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**model_inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].detach().cpu().tolist()

    word_labels = []
    previous_word_idx = None

    for pred_id, word_idx in zip(predictions, word_ids):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            word_labels.append(id2label[int(pred_id)])
        previous_word_idx = word_idx

    return list(zip(tokens, word_labels))

Функция для выделения сущностей из BIO

In [76]:
def extract_entities_from_bio(word_label_pairs):
    entities = []
    current_tokens = []
    current_type = None

    for token, label in word_label_pairs:
        if label == "O":
            if current_tokens:
                entities.append((" ".join(current_tokens), current_type))
                current_tokens = []
                current_type = None
            continue

        prefix, ent_type = label.split("-", 1)

        if prefix == "B":
            if current_tokens:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = [token]
            current_type = ent_type

        elif prefix == "I":
            if current_tokens and current_type == ent_type:
                current_tokens.append(token)
            else:
                # если вдруг I-* пришёл без корректного B-*, начинаем новую сущность
                if current_tokens:
                    entities.append((" ".join(current_tokens), current_type))
                current_tokens = [token]
                current_type = ent_type

    if current_tokens:
        entities.append((" ".join(current_tokens), current_type))

    return entities

Функция для красивого вывода

In [77]:
def show_prediction(model, tokenizer, tokens, id2label, title="Model prediction"):
    word_label_pairs = predict_word_labels(model, tokenizer, tokens, id2label)
    entities = extract_entities_from_bio(word_label_pairs)

    print(f"\n=== {title} ===")
    print("Text:")
    print(" ".join(tokens))

    print("\nWord-level labels:")
    for token, label in word_label_pairs:
        print(f"{token:20s} -> {label}")

    print("\nExtracted entities:")
    if len(entities) == 0:
        print("No entities found")
    else:
        for ent_text, ent_type in entities:
            print(f"{ent_text} -> {ent_type}")

In [79]:
def show_gold(tokens, true_tags, id2label):
    gold_pairs = [(tok, id2label[tag]) for tok, tag in zip(tokens, true_tags)]
    entities = extract_entities_from_bio(gold_pairs)

    print("\n=== Gold labels ===")
    print("Text:")
    print(" ".join(tokens))

    print("\nWord-level labels:")
    for token, label in gold_pairs:
        print(f"{token:20s} -> {label}")

    print("\nGold entities:")
    if len(entities) == 0:
        print("No entities found")
    else:
        for ent_text, ent_type in entities:
            print(f"{ent_text} -> {ent_type}")

Применение

In [73]:
sample = dataset["test"][0]
tokens = sample["tokens"]
true_tags = sample["ner_tags"]

print(tokens)
print(true_tags)

['Он', 'широко', 'расставил', 'ноги', ',', 'словно', 'пытаясь', 'обрести', 'равновесие', ',', 'отвернулся', 'от', 'ледяного', 'ветра', 'и', 'вцепился', 'в', 'отороченную', 'мехом', 'шапку', ',', 'моргая', 'на', 'Сэрен', 'Педак', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0]


In [78]:
gold_pairs = [(tok, id2label[tag]) for tok, tag in zip(tokens, true_tags)]
print(gold_pairs)
print(extract_entities_from_bio(gold_pairs))

[('Он', 'O'), ('широко', 'O'), ('расставил', 'O'), ('ноги', 'O'), (',', 'O'), ('словно', 'O'), ('пытаясь', 'O'), ('обрести', 'O'), ('равновесие', 'O'), (',', 'O'), ('отвернулся', 'O'), ('от', 'O'), ('ледяного', 'O'), ('ветра', 'O'), ('и', 'O'), ('вцепился', 'O'), ('в', 'O'), ('отороченную', 'O'), ('мехом', 'O'), ('шапку', 'O'), (',', 'O'), ('моргая', 'O'), ('на', 'O'), ('Сэрен', 'B-PER'), ('Педак', 'I-PER'), ('.', 'O')]
[('Сэрен Педак', 'PER')]


In [85]:
show_gold(tokens, true_tags, id2label)


=== Gold labels ===
Text:
Он широко расставил ноги , словно пытаясь обрести равновесие , отвернулся от ледяного ветра и вцепился в отороченную мехом шапку , моргая на Сэрен Педак .

Word-level labels:
Он                   -> O
широко               -> O
расставил            -> O
ноги                 -> O
,                    -> O
словно               -> O
пытаясь              -> O
обрести              -> O
равновесие           -> O
,                    -> O
отвернулся           -> O
от                   -> O
ледяного             -> O
ветра                -> O
и                    -> O
вцепился             -> O
в                    -> O
отороченную          -> O
мехом                -> O
шапку                -> O
,                    -> O
моргая               -> O
на                   -> O
Сэрен                -> B-PER
Педак                -> I-PER
.                    -> O

Gold entities:
Сэрен Педак -> PER


In [86]:
show_prediction(untrained_model, tokenizer, tokens, id2label, title="Base model")


=== Base model ===
Text:
Он широко расставил ноги , словно пытаясь обрести равновесие , отвернулся от ледяного ветра и вцепился в отороченную мехом шапку , моргая на Сэрен Педак .

Word-level labels:
Он                   -> I-LOC
широко               -> I-PER
расставил            -> I-PER
ноги                 -> I-PER
,                    -> B-LOC
словно               -> O
пытаясь              -> O
обрести              -> O
равновесие           -> O
,                    -> B-LOC
отвернулся           -> I-LOC
от                   -> I-LOC
ледяного             -> I-LOC
ветра                -> I-LOC
и                    -> B-LOC
вцепился             -> B-LOC
в                    -> B-LOC
отороченную          -> B-RACE
мехом                -> O
шапку                -> B-PER
,                    -> B-LOC
моргая               -> O
на                   -> O
Сэрен                -> B-PER
Педак                -> O
.                    -> I-PER

Extracted entities:
Он -> LOC
широко расставил но

In [87]:
show_prediction(trainer.model, tokenizer, tokens, id2label, title="Fine-tuned model")


=== Fine-tuned model ===
Text:
Он широко расставил ноги , словно пытаясь обрести равновесие , отвернулся от ледяного ветра и вцепился в отороченную мехом шапку , моргая на Сэрен Педак .

Word-level labels:
Он                   -> O
широко               -> O
расставил            -> O
ноги                 -> O
,                    -> O
словно               -> O
пытаясь              -> O
обрести              -> O
равновесие           -> O
,                    -> O
отвернулся           -> O
от                   -> O
ледяного             -> O
ветра                -> O
и                    -> O
вцепился             -> O
в                    -> O
отороченную          -> O
мехом                -> O
шапку                -> O
,                    -> O
моргая               -> O
на                   -> O
Сэрен                -> B-PER
Педак                -> I-PER
.                    -> O

Extracted entities:
Сэрен Педак -> PER
